# SC2001 Project 2 - Part (a): Dijkstra with an Adjacency Matrix and an Array Priority Queue

**Task.** The input graph $G = (V, E)$ is stored in an **adjacency matrix** and the
priority queue is an **array**. Implement Dijkstra's algorithm in this setting and
analyse its time complexity with respect to $|V|$ and $|E|$, both theoretically and empirically.

**Contents**
1. Implementation
2. Theoretical analysis
3. Correctness check
4. Empirical analysis: vary $|V|$, then vary $|E|$
5. Conclusion

## Setup

In [ ]:
%matplotlib inline

import gc
import csv
import math
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt

INF = math.inf
SEED = 2001        # fixes the random graphs so the results are reproducible
MAX_WEIGHT = 100   # edge weights are random integers in [1, MAX_WEIGHT]

QUICK = False      # True = small sizes for a fast run (~20 s); False = full run (~1-2 min)

OUTPUT_DIR = Path("part a output")   # every figure, table and printed result is saved here
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_FILE = OUTPUT_DIR / "a_results.txt"
LOG_FILE.write_text("", encoding="utf-8")   # start a fresh log on every run


def log(*args, **kwargs):
    # print() that also appends the same text to part a output/a_results.txt
    print(*args, **kwargs)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        print(*args, **kwargs, file=f)


def save_csv(name, header, rows):
    with open(OUTPUT_DIR / name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)

---
## 1. Implementation

**Data structures**

| Name | Meaning |
|---|---|
| `graph[u][v]` | weight of edge $(u, v)$, or `INF` if there is no edge ($|V| \times |V|$ matrix) |
| `d[v]` | current shortest-distance estimate of $v$; this array **is** the priority queue |
| `pi[v]` | predecessor of $v$ on the shortest path |
| `S[v]` | `True` once $v$'s distance is final, i.e. $v$ has been removed from the queue |

**Priority-queue operations on an array**
- *Extract-min*: scan all $|V|$ entries for the unvisited vertex with the smallest `d`, so $O(|V|)$.
- *Decrease-key*: overwrite `d[v]`, so $O(1)$.

**Finding the neighbours of $u$.** A matrix cannot list only $u$'s neighbours, so the whole
row `graph[u][0..|V|-1]` is scanned, which is $O(|V|)$ no matter how many edges $u$ has.

The function also counts its elementary operations. These counts are exact and free of
timing noise, so they test the analysis more directly than wall-clock time:
- `pq_scan`: array entries examined by extract-min
- `matrix_scan`: matrix cells examined while relaxing
- `decrease_key`: times a distance was improved

### Pseudocode with the cost of each line

```
DIJKSTRA-MATRIX-ARRAY(G, s)                                  cost      times
1  for each vertex v:  d[v] = INF, pi[v] = NIL, S[v] = False  O(1)      |V|
2  d[s] = 0                                                   O(1)      1
3  repeat |V| times:
4      u = unvisited vertex with the smallest d[u]            O(|V|)    |V|      <- extract-min: scan d[]
5      S[u] = True                                            O(1)      |V|
6      for v = 0 .. |V|-1:                                    O(1)      |V|*|V|  <- read row u of the matrix
7          if G[u][v] != INF and not S[v]
                  and d[u] + G[u][v] < d[v]:                  O(1)      |V|*|V|
8              d[v] = d[u] + G[u][v];  pi[v] = u              O(1)      <= |E|   <- decrease-key
```

Lines 4 and 6-7 each run $|V|^2$ times in total, whatever the graph. Line 8 runs once per
successful relaxation, and each edge $(u, v)$ is examined only once (when $u$ is extracted),
so it runs at most $|E|$ times.

In [ ]:
def dijkstra_matrix_array(graph, source):
    # Single-source shortest paths: adjacency matrix + array priority queue.
    # Returns (d, pi, ops), where ops counts the elementary operations.
    n = len(graph)
    d = [INF] * n          # the priority queue: d[v] is v's priority
    pi = [None] * n
    S = [False] * n        # S[v] = True means v has left the queue
    d[source] = 0

    ops = {"pq_scan": 0, "matrix_scan": 0, "decrease_key": 0}

    for _ in range(n):
        # Extract-min: linear scan over the array, O(|V|)
        u = -1
        best = INF
        for v in range(n):
            ops["pq_scan"] += 1
            if not S[v] and d[v] < best:
                best = d[v]
                u = v
        if u == -1:        # every remaining vertex is unreachable
            break
        S[u] = True

        # Relax every edge out of u: scan the whole matrix row, O(|V|)
        row = graph[u]
        for v in range(n):
            ops["matrix_scan"] += 1
            w = row[v]
            if w != INF and not S[v] and d[u] + w < d[v]:
                d[v] = d[u] + w          # decrease-key on an array is O(1)
                pi[v] = u
                ops["decrease_key"] += 1

    return d, pi, ops

### Illustration: why the matrix costs $|V|$ per vertex

The CLRS example graph (Fig. 24.6) next to its adjacency matrix. To find the neighbours of
$y$, the algorithm has to read the whole row for $y$, including the $\infty$ cells for
non-edges. Every vertex costs a full row of $|V|$ reads, whether it has one edge or $|V|-1$.

In [ ]:
def draw_example(highlight="y"):
    names = ["s", "t", "x", "y", "z"]
    xy = {"s": (0, 1), "t": (1.5, 2), "x": (3.5, 2), "y": (1.5, 0), "z": (3.5, 0)}
    edges = [("s", "t", 10), ("s", "y", 5), ("t", "x", 1), ("t", "y", 2), ("x", "z", 4),
             ("y", "t", 3), ("y", "x", 9), ("y", "z", 2), ("z", "s", 7), ("z", "x", 6)]
    hl_color, rad = "tab:orange", 0.15

    fig, (ax_g, ax_m) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.1, 1]})

    # Graph: curved arrows so that opposite edges (t<->y, x<->z) do not overlap
    for a, b, w in edges:
        (x1, y1), (x2, y2) = xy[a], xy[b]
        color = hl_color if a == highlight else "dimgray"
        ax_g.annotate("", xy=(x2, y2), xytext=(x1, y1),
                      arrowprops=dict(arrowstyle="-|>", color=color, lw=2 if a == highlight else 1.3,
                                      shrinkA=17, shrinkB=17, mutation_scale=16,
                                      connectionstyle=f"arc3,rad={rad}"))
        # midpoint of matplotlib's arc3 Bezier curve
        mx = (x1 + x2) / 2 + 0.5 * rad * (y2 - y1)
        my = (y1 + y2) / 2 - 0.5 * rad * (x2 - x1)
        ax_g.text(mx, my, str(w), ha="center", va="center", fontsize=11, color=color,
                  bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))
    for name, (x, y) in xy.items():
        ax_g.add_patch(plt.Circle((x, y), 0.22, fc=hl_color if name == highlight else "white",
                                  ec="black", lw=1.5, zorder=3))
        ax_g.text(x, y, name, ha="center", va="center", fontsize=14, zorder=4)
    ax_g.set_xlim(-0.5, 4)
    ax_g.set_ylim(-0.5, 2.5)
    ax_g.set_aspect("equal")
    ax_g.axis("off")
    ax_g.set_title("Example graph G (CLRS Fig. 24.6)")

    # Adjacency matrix with the highlighted row
    weight = {(a, b): w for a, b, w in edges}
    n = len(names)
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            row_hl = a == highlight
            ax_m.add_patch(plt.Rectangle((j, n - 1 - i), 1, 1, ec="black",
                                         fc=(hl_color if row_hl else "white"),
                                         alpha=0.35 if row_hl else 1))
            w = weight.get((a, b))
            ax_m.text(j + 0.5, n - 0.5 - i, "∞" if w is None else str(w), ha="center", va="center",
                      fontsize=13, color="black" if w is not None else "gray")
        ax_m.text(-0.3, n - 0.5 - i, a, ha="center", va="center", fontsize=13, weight="bold")
        ax_m.text(i + 0.5, n + 0.3, names[i], ha="center", va="center", fontsize=13, weight="bold")
    ax_m.text(n + 0.15, n - 0.5 - names.index(highlight),
              f"neighbours of {highlight}:\nread all {n} cells,\nincluding the ∞ ones",
              ha="left", va="center", fontsize=10, color="black")
    ax_m.set_xlim(-0.7, n + 2.6)
    ax_m.set_ylim(-0.2, n + 0.7)
    ax_m.set_aspect("equal")
    ax_m.axis("off")
    ax_m.set_title("Adjacency matrix G[u][v]  (row = from, column = to)")

    fig.tight_layout()
    return fig


fig = draw_example()
fig.savefig(OUTPUT_DIR / "a_example_graph_matrix.png", dpi=200)
plt.show()

### Generating random graphs

`random_graph_edges(n, m)` returns exactly $m$ distinct directed edges on $n$ vertices,
with no self-loops. The first $n-1$ edges form a random tree rooted at vertex 0, so every
vertex is reachable from the source and Dijkstra always processes all $n$ vertices. The
remaining edges are sampled uniformly from the $n(n-1)$ possible pairs, so any $m$ from
$n-1$ (a tree) to $n(n-1)$ (a complete graph) can be generated.

In [ ]:
def random_graph_edges(n, m, rng):
    max_edges = n * (n - 1)
    if not n - 1 <= m <= max_edges:
        raise ValueError(f"need {n - 1} <= m <= {max_edges}, got {m}")

    # Random spanning tree rooted at 0: each vertex gets a parent placed before it
    order = list(range(1, n))
    rng.shuffle(order)
    order = [0] + order
    pairs = set()
    for i in range(1, n):
        pairs.add((order[rng.randrange(i)], order[i]))

    # Sample the remaining edges by index into the n(n-1) off-diagonal pairs.
    # At most n-1 samples can collide with tree edges, so oversample by n-1.
    extra = m - (n - 1)
    k = min(max_edges, extra + n - 1)
    for idx in rng.sample(range(max_edges), k):
        if len(pairs) == m:
            break
        u, r = divmod(idx, n - 1)
        v = r if r < u else r + 1     # skip the diagonal (no self-loops)
        pairs.add((u, v))

    return [(u, v, rng.randint(1, MAX_WEIGHT)) for u, v in pairs]


def to_matrix(n, edges):
    graph = [[INF] * n for _ in range(n)]
    for u, v, w in edges:
        graph[u][v] = w
    return graph

---
## 2. Theoretical analysis

| Step | Work per execution | Times executed | Total |
|---|---|---|---|
| Initialise `d`, `pi`, `S` | $O(1)$ per vertex | $|V|$ | $O(|V|)$ |
| Extract-min (scan the array) | $O(|V|)$ | $|V|$ | $O(|V|^2)$ |
| Scan row `graph[u]` for neighbours | $O(|V|)$ | $|V|$ | $O(|V|^2)$ |
| Decrease-key (overwrite `d[v]`) | $O(1)$ | at most $|E|$ | $O(|E|)$ |

$$
T(|V|, |E|) = O(|V|) + O(|V|^2) + O(|V|^2) + O(|E|) = O(|V|^2 + |E|) = O(|V|^2)
$$

because a simple directed graph has $|E| \le |V|(|V|-1) < |V|^2$.

**The bound is tight in every case.** Both $|V|^2$ terms are paid in full whatever the
input: each extract-min scans all $|V|$ array entries, and each row scan reads all $|V|$
cells, including the empty ones. So the running time is $\Theta(|V|^2)$ in the best,
average and worst case. Exactly $|V|^2$ array entries and $|V|^2$ matrix cells are
examined, $2|V|^2$ in total.

**Dependence on $|E|$.** $|E|$ only affects how many decrease-keys happen, and each costs
$O(1)$. The cost is therefore asymptotically independent of $|E|$: a tree with $|V|-1$ edges
costs the same $\Theta(|V|^2)$ as a complete graph with $|V|(|V|-1)$ edges.

**Predictions to test empirically**
1. Running time grows as $|V|^2$, so time $/\,|V|^2$ stays roughly constant.
2. The operation count `pq_scan + matrix_scan` is exactly $2|V|^2$.
3. With $|V|$ fixed, changing $|E|$ leaves the operation count unchanged and the time nearly flat.

### Where the $2|V|^2$ comes from

Left: one round reads all of `d[ ]` (extract-min) and all of one matrix row (relaxing $u$),
so $2|V|$ cells. Right: after all $|V|$ rounds, `d[ ]` has been scanned $|V|$ times and every
matrix row has been read exactly once, giving two full $|V| \times |V|$ grids: $2|V|^2$ reads,
whatever the number of edges.

In [ ]:
def draw_cost_diagram(n=8, u=3):
    blue, orange, grey = "tab:blue", "tab:orange", "#9a9a9a"
    rng = random.Random(7)
    weights = [[None if i == j or rng.random() > 0.35 else rng.randint(1, 9) for j in range(n)]
               for i in range(n)]
    d_vals = ["0", "4", "∞", "2", "7", "∞", "5", "∞"]
    done = [True, False, False, True, False, False, False, False]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.2), gridspec_kw={"width_ratios": [1, 1.15]})

    # Panel 1: one round
    for j in range(n):
        ax1.add_patch(plt.Rectangle((j, 10), 1, 1, ec="black",
                                    fc="#dddddd" if done[j] else blue, alpha=0.9 if done[j] else 0.3))
        ax1.text(j + 0.5, 10.5, d_vals[j], ha="center", va="center", fontsize=17)
    ax1.text(-0.25, 10.5, "d[ ]", ha="right", va="center", fontsize=19, weight="bold")
    ax1.annotate("", xy=(n, 11.45), xytext=(0, 11.45),
                 arrowprops=dict(arrowstyle="-|>", color=blue, lw=2.2))
    ax1.text(n / 2, 11.8, "extract-min: scan all |V| entries", ha="center", fontsize=18, color=blue)
    for i in range(n):
        for j in range(n):
            row = i == u
            ax1.add_patch(plt.Rectangle((j, n - 1 - i), 1, 1, ec="black",
                                        fc=orange if row else "white", alpha=0.4 if row else 1))
            w = weights[i][j]
            ax1.text(j + 0.5, n - 0.5 - i, "∞" if w is None else str(w), ha="center", va="center",
                     fontsize=15, color="black" if w is not None else grey)
    ax1.text(-0.25, n / 2, "matrix", ha="right", va="center", fontsize=19, weight="bold")
    ax1.text(n + 0.3, n - 0.5 - u, "← row u: read all |V| cells,\n     ∞ ones included", ha="left",
             va="center", fontsize=18, color="#b35900")
    ax1.set_title("One round  =  2|V| reads", fontsize=23, pad=14)
    ax1.set_xlim(-2.0, n + 8.2)
    ax1.set_ylim(-0.5, 12.5)

    # Panel 2: all rounds
    for k, (x0, color, label) in enumerate([(0, blue, "d[ ] scanned every round\n= |V|² reads"),
                                            (n + 4, orange, "each matrix row read once\n= |V|² reads")]):
        for i in range(n):
            for j in range(n):
                ax2.add_patch(plt.Rectangle((x0 + j, n - 1 - i), 1, 1, ec="white", fc=color, alpha=0.75))
        ax2.text(x0 + n / 2, -1.3, label, ha="center", va="center", fontsize=18)
    ax2.text(n + 2, n / 2, "+", ha="center", va="center", fontsize=34, weight="bold")
    ax2.text(-0.4, n / 2, "|V| rounds", ha="right", va="center", fontsize=18, rotation=90)
    ax2.set_title("All |V| rounds  =  2|V|² reads  →  Θ(|V|²), for any |E|", fontsize=23, pad=14)
    ax2.set_xlim(-1.4, 2 * n + 4.4)
    ax2.set_ylim(-3.0, n + 0.8)

    for ax in (ax1, ax2):
        ax.set_aspect("equal")
        ax.axis("off")
    fig.tight_layout()
    return fig


fig = draw_cost_diagram()
fig.savefig(OUTPUT_DIR / "a_cost_diagram.png", dpi=200, bbox_inches="tight", transparent=True)
plt.show()

---
## 3. Correctness check

1. The textbook example from CLRS (Fig. 24.6), with vertices $s, t, x, y, z = 0, 1, 2, 3, 4$
2. A graph with an unreachable vertex, whose distance must stay $\infty$
3. 200 random graphs, checked against Bellman-Ford, an independent $O(|V||E|)$ algorithm

In [ ]:
def bellman_ford(n, edges, source):
    # Independent reference algorithm, used only to check the answers
    d = [INF] * n
    d[source] = 0
    for _ in range(n - 1):
        changed = False
        for u, v, w in edges:
            if d[u] + w < d[v]:
                d[v] = d[u] + w
                changed = True
        if not changed:
            break
    return d


# 1. Textbook example
edges = [(0, 1, 10), (0, 3, 5), (1, 2, 1), (1, 3, 2), (2, 4, 4),
         (3, 1, 3), (3, 2, 9), (3, 4, 2), (4, 0, 7), (4, 2, 6)]
d, pi, _ = dijkstra_matrix_array(to_matrix(5, edges), 0)
log("textbook example  d  =", d)
log("                  pi =", pi)
assert d == [0, 8, 9, 5, 7]
assert pi == [None, 3, 1, 0, 3]

# 2. Unreachable vertex
d, _, _ = dijkstra_matrix_array(to_matrix(3, [(0, 1, 4)]), 0)
log("unreachable case  d  =", d)
assert d == [0, 4, INF]

# 3. Random graphs against Bellman-Ford
rng = random.Random(SEED)
for _ in range(200):
    n = rng.randint(2, 40)
    m = rng.randint(n - 1, n * (n - 1))
    edges = random_graph_edges(n, m, rng)
    d, _, _ = dijkstra_matrix_array(to_matrix(n, edges), 0)
    assert d == bellman_ford(n, edges, 0)

log("\nAll correctness checks pass (textbook example, unreachable case, 200 random graphs).")

---
## 4. Empirical analysis

### Experimental setup

| Item | Full run (`QUICK = False`) |
|---|---|
| Graphs | Random directed graphs, connected from source 0: a random spanning tree plus extra edges sampled uniformly, giving exactly the requested $|E|$ |
| Edge weights | Uniform random integers in $[1, 100]$ |
| Source | Vertex 0 |
| Experiment 1 (vary $|V|$) | $|V| = 200, 400, \ldots, 2000$ at two densities: sparse $|E| = 5|V|$ and dense $|E| = |V|(|V|-1)/2$ |
| Experiment 2 (vary $|E|$) | $|V| = 1500$ fixed; 9 values of $|E|$ from $1{,}499$ (a tree) to $2{,}248{,}500$ (complete) |
| Timing | `time.perf_counter`, fastest of 5 interleaved passes, garbage collector paused, graph building not timed |
| Operation counts | Exact counts from the instrumented implementation |
| Random seed | 2001, so every run uses identical graphs |

`measure_all` builds every random graph for an experiment first; building them is not
timed. It then makes `PASSES` passes, timing Dijkstra once on every graph in each pass,
and keeps each graph's fastest time.

Background programs and power-saving can slow the machine for several seconds at a time.
Running each graph's repetitions back to back would let one such slowdown cover all of
them. Interleaving means a slowdown only hits the graphs running during it, in that one
pass, and taking the minimum discards it. The garbage collector is also paused while
timing so its pauses don't add noise.

`log_slope` fits $\log(\text{time}) = k \log|V| + c$ by least squares. The slope $k$ is the
measured exponent in time $\propto |V|^k$, and the theory predicts $k \approx 2$.

In [ ]:
def measure_all(configs, passes, rng):
    # configs: list of (n, m). Returns [(best time, ops)] in the same order.
    # Each pass times every graph once, so a burst of background load
    # only slows the graphs measured during that burst, in that one pass.
    graphs = [to_matrix(n, random_graph_edges(n, m, rng)) for n, m in configs]
    best = [INF] * len(configs)
    ops = [None] * len(configs)
    gc.disable()
    try:
        for _ in range(passes):
            for i, graph in enumerate(graphs):
                start = time.perf_counter()
                _, _, ops[i] = dijkstra_matrix_array(graph, 0)
                best[i] = min(best[i], time.perf_counter() - start)
    finally:
        gc.enable()
    return list(zip(best, ops))


def save_panels(panels, combined_name, combined_size):
    # Draws every panel side by side (saved as combined_name), and also saves each
    # panel as its own PNG so it can be used on a slide by itself.
    fig, axes = plt.subplots(1, len(panels), figsize=combined_size)
    for (draw, _), ax in zip(panels, axes):
        draw(ax)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / combined_name, dpi=150)
    plt.show()
    for draw, name in panels:
        single, ax = plt.subplots(figsize=(6.4, 4.6))
        draw(ax)
        single.tight_layout()
        single.savefig(OUTPUT_DIR / name, dpi=200)
        plt.close(single)


def log_slope(xs, ys):
    lx = [math.log(x) for x in xs]
    ly = [math.log(y) for y in ys]
    mx, my = sum(lx) / len(lx), sum(ly) / len(ly)
    num = sum((a - mx) * (b - my) for a, b in zip(lx, ly))
    den = sum((a - mx) ** 2 for a in lx)
    return num / den


if QUICK:
    SIZES, PASSES, FIXED_V, E_STEPS = list(range(100, 801, 100)), 3, 500, 6
else:
    SIZES, PASSES, FIXED_V, E_STEPS = list(range(200, 2001, 200)), 5, 1500, 8

log(f"Python {platform.python_version()} on {platform.platform()}")
log(f"Sizes {SIZES[0]}..{SIZES[-1]}, fixed |V| = {FIXED_V}, {PASSES} passes, QUICK = {QUICK}")

### 4.1 Varying $|V|$

$|V|$ runs over a range of sizes at two densities:
- **sparse**: $|E| = 5|V|$
- **dense**: $|E| = |V|(|V|-1)/2$, half of all possible edges

If the analysis is right, both densities grow as $|V|^2$ and take about the same time.

In [ ]:
rng = random.Random(SEED)
vary_v = {"sparse": [], "dense": []}

header = (f"{'|V|':>6} {'density':>7} {'|E|':>9} {'time (s)':>9} {'t/|V|^2 (ns)':>13} "
          f"{'pq_scan':>10} {'matrix_scan':>11} {'dec_key':>8}")
log(header)
log("-" * len(header))
labelled = [(label, n, m) for n in SIZES
            for label, m in (("sparse", 5 * n), ("dense", n * (n - 1) // 2))]
results = measure_all([(n, m) for _, n, m in labelled], PASSES, rng)
for (label, n, m), (t, ops) in zip(labelled, results):
    vary_v[label].append((n, m, t, ops))
    log(f"{n:>6} {label:>7} {m:>9} {t:>9.4f} {t / n**2 * 1e9:>13.2f} "
          f"{ops['pq_scan']:>10} {ops['matrix_scan']:>11} {ops['decrease_key']:>8}")

log()
for label, rows in vary_v.items():
    k = log_slope([r[0] for r in rows], [r[2] for r in rows])
    exact = all(r[3]["pq_scan"] + r[3]["matrix_scan"] == 2 * r[0] ** 2 for r in rows)
    log(f"{label:>6}: fitted exponent k = {k:.2f};  scans == 2|V|^2 for every size: {exact}")

save_csv("a_vary_V.csv",
         ["V", "density", "E", "time_s", "time_per_V2_ns", "pq_scan", "matrix_scan", "decrease_key"],
         [(n, label, m, t, t / n ** 2 * 1e9, o["pq_scan"], o["matrix_scan"], o["decrease_key"])
          for label, rows in vary_v.items() for n, m, t, o in rows])

In [ ]:
def fit_c_v2(rows):
    # Least-squares fit of time = c * |V|^2 through the origin; returns (c, R^2)
    xs = [r[0] ** 2 for r in rows]
    ts = [r[2] for r in rows]
    c = sum(x * t for x, t in zip(xs, ts)) / sum(x * x for x in xs)
    mean_t = sum(ts) / len(ts)
    ss_res = sum((t - c * x) ** 2 for x, t in zip(xs, ts))
    ss_tot = sum((t - mean_t) ** 2 for t in ts)
    return c, 1 - ss_res / ss_tot


fits = {label: fit_c_v2(rows) for label, rows in vary_v.items()}
for label, (c, r2) in fits.items():
    log(f"{label:>6}: time = {c * 1e9:.0f} ns x |V|^2,  R^2 = {r2:.4f}")

STYLE = {"sparse": ("o", "tab:blue"), "dense": ("s", "tab:orange")}


def panel_time_vs_v(ax):
    for label, rows in vary_v.items():
        marker, color = STYLE[label]
        ns = [r[0] for r in rows]
        c, r2 = fits[label]
        smooth = range(ns[0], ns[-1] + 1, 10)
        ax.plot(ns, [r[2] for r in rows], marker=marker, ls="none", color=color, label=f"{label}: measured")
        ax.plot(smooth, [c * n * n for n in smooth], color=color,
                label=f"{label}: fit {c * 1e9:.0f} ns $\\times$ |V|$^2$ (R$^2$ = {r2:.3f})")
    ax.set(title="Running time vs |V|, with fitted c|V|$^2$", xlabel="|V|", ylabel="time (s)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)


def panel_normalised(ax):
    for label, rows in vary_v.items():
        marker, color = STYLE[label]
        ax.plot([r[0] for r in rows], [r[2] / r[0] ** 2 * 1e9 for r in rows],
                marker=marker, color=color, label=f"{label} graph")
    ax.set(title="time / |V|$^2$  (flat line = $\\Theta(|V|^2)$)", xlabel="|V|", ylabel="ns per |V|$^2$")
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(alpha=0.3)


def panel_ops_vs_v(ax):
    ns = [r[0] for r in vary_v["dense"]]
    for label, style in (("dense", "s-"), ("sparse", "o--")):
        ax.plot(ns, [r[3]["pq_scan"] + r[3]["matrix_scan"] for r in vary_v[label]], style,
                color=STYLE[label][1], label=f"cells scanned ({label})")
    ax.plot(ns, [2 * n * n for n in ns], ":", color="black", label="2|V|$^2$")
    ax.set(title="Operation count vs |V|", xlabel="|V|", ylabel="operations")
    ax.legend()
    ax.grid(alpha=0.3)


save_panels([(panel_time_vs_v, "a_V_time.png"),
             (panel_normalised, "a_V_normalised.png"),
             (panel_ops_vs_v, "a_V_ops.png")],
            "a_vary_V.png", (17, 4.8))

**Observations**
- **Time grows quadratically.** The fitted exponents are close to 2, and a single-parameter
  curve $c|V|^2$ fits the measured times with $R^2$ close to 1 (left plot). Time $/\,|V|^2$
  stays roughly flat across sizes (middle plot), which is the signature of $\Theta(|V|^2)$:
  a faster-growing cost would make it rise, a slower-growing one would make it fall.
- **Operation counts match exactly.** `pq_scan` and `matrix_scan` are each exactly $|V|^2$,
  so their sum is $2|V|^2$ for both densities. The sparse and dense curves lie on top of
  each other and on the $2|V|^2$ line.
- **Decrease-key is negligible.** Even on dense graphs it happens only a few thousand
  times, compared with millions of scans.
- **Dense graphs are slightly slower, by a constant factor only.** When a cell is empty,
  the condition `w != INF` fails immediately and the loop moves on. When there is an edge,
  it also checks `S[v]` and computes `d[u] + w`. Each cell therefore costs a little more on a
  dense graph. This raises the constant in front of $|V|^2$ (the two flat lines in the
  middle plot sit at different heights) but not the growth rate.

### 4.2 Varying $|E|$ with $|V|$ fixed

$|V|$ is held fixed while $|E|$ goes from $|V|-1$ (a tree) to $|V|(|V|-1)$ (a complete
directed graph). The theory predicts the same operation count throughout and a nearly
flat running time.

**Paired timing.** This experiment asks a relative question: does adding edges change the
time? Every graph here takes about a second, and over a run of a minute or more a laptop's
speed can drift by more than the effect being measured (CPU clock and power-state changes).
So each graph is timed **right after the tree graph** ($|E| = |V|-1$), and the result is
the ratio of the two times. Any slowdown of the machine affects both runs of a pair almost
equally and cancels out. The order inside each pair alternates between passes, and each
graph's ratio is the median over all passes. A ratio of 1 means "as fast as the tree";
the theory predicts ratios that stay close to 1.

In [ ]:
def measure_relative(n, edge_counts, passes, rng):
    # Times every graph paired with the tree graph (the first entry, |E| = n-1).
    # Returns [(median time ratio, ops)] in the order of edge_counts.
    graphs = [to_matrix(n, random_graph_edges(n, m, rng)) for m in edge_counts]
    tree = graphs[0]

    def run(graph):
        start = time.perf_counter()
        _, _, ops = dijkstra_matrix_array(graph, 0)
        return time.perf_counter() - start, ops

    ratios = [[] for _ in graphs]
    ops = [None] * len(graphs)
    gc.disable()
    try:
        for p in range(passes):
            for i, graph in enumerate(graphs):
                if p % 2 == 0:                     # alternate the order inside each pair
                    t_tree, _ = run(tree)
                    t, ops[i] = run(graph)
                else:
                    t, ops[i] = run(graph)
                    t_tree, _ = run(tree)
                ratios[i].append(t / t_tree)
    finally:
        gc.enable()
    medians = [sorted(r)[len(r) // 2] for r in ratios]
    return list(zip(medians, ops))


rng = random.Random(SEED + 1)
max_m = FIXED_V * (FIXED_V - 1)
edge_counts = sorted({FIXED_V - 1} | {round(max_m * i / E_STEPS) for i in range(1, E_STEPS + 1)})
vary_e = []

log(f"|V| = {FIXED_V} fixed\n")
header = (f"{'|E|':>9} {'density':>8} {'time / tree':>12} "
          f"{'pq_scan':>10} {'matrix_scan':>11} {'dec_key':>8}")
log(header)
log("-" * len(header))
results = measure_relative(FIXED_V, edge_counts, PASSES, rng)
for m, (ratio, ops) in zip(edge_counts, results):
    vary_e.append((m, ratio, ops))
    log(f"{m:>9} {m / max_m:>8.1%} {ratio:>12.3f} {ops['pq_scan']:>10} "
        f"{ops['matrix_scan']:>11} {ops['decrease_key']:>8}")

save_csv("a_vary_E.csv",
         ["V", "E", "density", "time_relative_to_tree", "pq_scan", "matrix_scan", "decrease_key"],
         [(FIXED_V, m, m / max_m, ratio, o["pq_scan"], o["matrix_scan"], o["decrease_key"])
          for m, ratio, o in vary_e])

# Straight-line fit of the ratio: ratio = a + b|E|
ratios = [r[1] for r in vary_e]
mean_e = sum(edge_counts) / len(edge_counts)
mean_r = sum(ratios) / len(ratios)
b = (sum((e - mean_e) * (r - mean_r) for e, r in zip(edge_counts, ratios))
     / sum((e - mean_e) ** 2 for e in edge_counts))
a = mean_r - b * mean_e
log(f"\n|E| grew by a factor of {edge_counts[-1] / edge_counts[0]:.0f}")
log(f"straight-line fit: time / tree = {a:.3f} + {b * max_m:.3f} x (|E| / |V|(|V|-1))")
log(f"  a complete graph takes {a + b * max_m:.2f}x as long as a tree on the same {FIXED_V} vertices")

In [ ]:
es = [r[0] for r in vary_e]


def panel_time_vs_e(ax):
    ax.plot(es, [r[1] for r in vary_e], "o", label="measured (median over passes)")
    ax.plot(es, [a + b * e for e in es], ":", color="black",
            label=f"straight-line fit: {a:.2f} to {a + b * max_m:.2f}")
    ax.axhline(1, color="gray", lw=0.8)
    ax.set(title=f"Running time vs |E|, relative to a tree  (|V| = {FIXED_V})",
           xlabel="|E|", ylabel="time / time of the tree graph")
    ax.set_ylim(0, max(2, max(r[1] for r in vary_e) * 1.15))
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)


def panel_ops_vs_e(ax):
    ax.plot(es, [r[2]["pq_scan"] + r[2]["matrix_scan"] for r in vary_e], "o-", label="cells scanned")
    ax.plot(es, [r[2]["decrease_key"] for r in vary_e], "s-", label="decrease-key")
    ax.axhline(2 * FIXED_V ** 2, ls=":", color="black", label="2|V|$^2$")
    ax.set(title=f"Operation count vs |E|  (|V| = {FIXED_V})", xlabel="|E|", ylabel="operations (log scale)")
    ax.set_yscale("log")
    ax.legend()
    ax.grid(alpha=0.3)


save_panels([(panel_time_vs_e, "a_E_time.png"),
             (panel_ops_vs_e, "a_E_ops.png")],
            "a_vary_E.png", (13, 4.6))

**Observations**
- **The scan count does not depend on $|E|$.** It is exactly $2|V|^2$ for every value of
  $|E|$, from a tree up to a complete graph.
- **Time is almost flat.** $|E|$ grows by three orders of magnitude, from a tree to a
  complete graph, but the time rises by well under 2× (see the printed fit).
- **The small rise has a clear cause.** A cell that holds an edge costs a little more than
  an empty one: the test `w != INF` passes, so the algorithm goes on to check `S[v]` and
  compute `d[u] + w`. Since every cell is read exactly once either way, this extra cost is
  bounded by a constant factor, reached when every cell holds an edge. $|E|$ changes the
  constant in front of $|V|^2$, never the growth rate.
- **Decrease-keys level off.** Their count rises quickly, then plateaus at a few thousand,
  far below $|E|$. A vertex's distance is improved only a few times on average, and the
  number of improvements is capped at $|E|$ by the theory.

---
## 5. Conclusion

With an adjacency matrix and an array priority queue, Dijkstra's algorithm runs in

$$\Theta(|V|^2)$$

in the best, average and worst case, independently of $|E|$.

- **Theory.** There are $|V|$ extract-mins, each an $O(|V|)$ array scan, and $|V|$ row
  scans, each $O(|V|)$. That gives $2|V|^2$ cell reads. The $\le |E|$ decrease-keys add
  only $O(|E|) \subseteq O(|V|^2)$.
- **Experiments.**
  - The operation count is exactly $2|V|^2$ for every graph tested.
  - The measured running-time exponent is about 2.
  - Holding $|V|$ fixed while increasing $|E|$ by three orders of magnitude changes the
    time only by a small constant factor.

**Implication.** This implementation pays the full $|V|^2$ even on a very sparse graph, but
nothing extra on a complete one. It therefore suits dense graphs, where $|E| = \Theta(|V|^2)$
and no algorithm can avoid reading the edges anyway. On sparse graphs most of its work is
spent scanning empty matrix cells.

### How much of the work is useful?

Every cell of the matrix is read once, but only the cells that hold an edge can ever
change a distance. At the largest size tested, $|V| = 2000$:

In [ ]:
def draw_wasted_reads(n=2000):
    cases = [("sparse\n|E| = 5|V|", 5 * n), ("dense\n|E| = |V|(|V|−1)/2", n * (n - 1) // 2)]
    fig, ax = plt.subplots(figsize=(12, 4.2))
    for i, (label, m) in enumerate(cases):
        useful = 100 * m / n ** 2           # share of the |V|^2 matrix cells that hold an edge
        ax.barh(i, useful, color="tab:orange", height=0.55,
                label="cell holds an edge (useful read)" if i == 0 else None)
        ax.barh(i, 100 - useful, left=useful, color="#d9d9d9", height=0.55,
                label="cell is ∞ (wasted read)" if i == 0 else None)
        ax.text(min(useful, 100) + 1, i, f"{useful:.2f}% hold an edge", va="center", fontsize=21,
                weight="bold", color="#7a3d00")
    ax.set_yticks([0, 1], [c[0] for c in cases], fontsize=19)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("% of matrix cells read  (grey = ∞ cell, a wasted read)", fontsize=17)
    ax.tick_params(axis="x", labelsize=15)
    ax.set_title(f"Matrix cells read by part (a) at |V| = {n}", fontsize=23, pad=14)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()
    return fig


fig = draw_wasted_reads()
fig.savefig(OUTPUT_DIR / "a_wasted_reads.png", dpi=200, bbox_inches="tight", transparent=True)
plt.show()